# VisionLab — Electronics + IoT YOLO Training

This notebook downloads two Roboflow datasets, converts their annotations into one compatible YOLO detection dataset, trains a lightweight YOLO11n model, and exports only the final `best.pt` checkpoint.

## Before you start

1. In Google Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Run each cell from top to bottom.
3. Enter your Roboflow API key only in the hidden prompt. Do not commit the key to GitHub.
4. Training uses a quick baseline preset (3 epochs, 320-pixel images, 25% of the training split). This does not guarantee high accuracy.


In [ ]:
!pip -q install roboflow ultralytics pyyaml

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not enabled. Select a GPU runtime before training.")


In [ ]:
from getpass import getpass
from roboflow import Roboflow

api_key = getpass("Enter your Roboflow API key: ")
rf = Roboflow(api_key=api_key)
print("Roboflow connection initialized.")


## 1. Download both datasets

The version numbers match the versions used when this notebook was prepared. If Roboflow reports that a version is unavailable or access is denied, check the project page and update the version number below.


In [ ]:
from pathlib import Path
import shutil

dataset_root = Path("/content/datasets")
dataset_root.mkdir(parents=True, exist_ok=True)

# Remove previous downloads so rerunning this cell starts cleanly.
for name in ("electronics", "iot"):
    folder = dataset_root / name
    if folder.exists():
        shutil.rmtree(folder)

general = (
    rf.workspace("vishnupriyas-workspace")
      .project("electronics-components-sveck")
      .version(7)
      .download("yolov8", location=str(dataset_root / "electronics"))
)

iot = (
    rf.workspace("design-project-zeico")
      .project("iot_dataset")
      .version(2)
      .download("yolov8", location=str(dataset_root / "iot"))
)

print("Electronics dataset:", general.location)
print("IoT dataset:", iot.location)


## 2. Merge the datasets and normalize annotations

YOLO detection labels have five fields: class ID plus normalized box coordinates. Some source images use YOLO polygon/segmentation labels instead; this cell converts those polygons to bounding boxes so both sources can train a standard detection model. This conversion does not preserve segmentation masks.


In [ ]:
import math
import yaml

sources = [
    ("electronics", Path(general.location)),
    ("iot", Path(iot.location)),
]

output = Path("/content/combined_components")
if output.exists():
    shutil.rmtree(output)

for split in ("train", "val", "test"):
    (output / split / "images").mkdir(parents=True, exist_ok=True)
    (output / split / "labels").mkdir(parents=True, exist_ok=True)

def normalize_class_name(name):
    key = str(name).strip().lower().replace("_", " ").replace("-", " ")
    key = " ".join(key.split())
    aliases = {
        "leds": "led",
        "led light": "led",
        "variable resistor": "potentiometer",
        "jumper wires": "jumper wire",
        "integrated circuit": "ic",
        "ic chip": "ic",
    }
    return aliases.get(key, key)

class_to_id = {}
class_display_names = {}
dataset_info = []

for prefix, root in sources:
    yaml_path = root / "data.yaml"
    if not yaml_path.is_file():
        raise FileNotFoundError(f"Dataset configuration not found: {yaml_path}")

    config = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))
    names = config["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=lambda value: int(value))]

    mapping = {}
    for old_id, display_name in enumerate(names):
        key = normalize_class_name(display_name)
        if not key:
            raise ValueError(f"Empty class name in {yaml_path}")
        if key not in class_to_id:
            class_to_id[key] = len(class_to_id)
            class_display_names[key] = str(display_name).strip()
        mapping[old_id] = class_to_id[key]

    dataset_info.append((prefix, root, mapping))

def convert_label_line(line, mapping, label_file):
    parts = line.split()
    if not parts:
        return None
    old_id = int(float(parts[0]))
    if old_id not in mapping:
        raise ValueError(f"Unknown class ID {old_id} in {label_file}")

    coordinates = [float(value) for value in parts[1:]]
    if not all(math.isfinite(value) for value in coordinates):
        raise ValueError(f"Non-finite coordinates in {label_file}")

    if len(coordinates) == 4:
        xc, yc, width, height = coordinates
        xmin, xmax = xc - width / 2, xc + width / 2
        ymin, ymax = yc - height / 2, yc + height / 2
    elif len(coordinates) >= 6 and len(coordinates) % 2 == 0:
        xs, ys = coordinates[0::2], coordinates[1::2]
        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)
        xc, yc = (xmin + xmax) / 2, (ymin + ymax) / 2
        width, height = xmax - xmin, ymax - ymin
    else:
        raise ValueError(f"Unsupported YOLO label format in {label_file}: {line[:120]}")

    xmin = min(1.0, max(0.0, xmin))
    xmax = min(1.0, max(0.0, xmax))
    ymin = min(1.0, max(0.0, ymin))
    ymax = min(1.0, max(0.0, ymax))
    width, height = xmax - xmin, ymax - ymin
    if width <= 0 or height <= 0:
        return None
    xc, yc = (xmin + xmax) / 2, (ymin + ymax) / 2

    return f"{mapping[old_id]} {xc:.8f} {yc:.8f} {width:.8f} {height:.8f}"

split_candidates = {
    "train": ("train",),
    "val": ("valid", "val"),
    "test": ("test",),
}

image_count = {split: 0 for split in split_candidates}
label_count = 0

for prefix, root, mapping in dataset_info:
    for out_split, candidates in split_candidates.items():
        source_split = next(
            (
                root / candidate
                for candidate in candidates
                if (root / candidate / "images").is_dir()
            ),
            None,
        )
        if source_split is None:
            continue

        source_images = source_split / "images"
        source_labels = source_split / "labels"
        destination_images = output / out_split / "images"
        destination_labels = output / out_split / "labels"

        for image_path in source_images.iterdir():
            if not image_path.is_file():
                continue

            new_name = f"{prefix}_{image_path.name}"
            shutil.copy2(image_path, destination_images / new_name)
            source_label = source_labels / f"{image_path.stem}.txt"
            destination_label = destination_labels / f"{Path(new_name).stem}.txt"

            converted_lines = []
            if source_label.is_file():
                for line in source_label.read_text(encoding="utf-8").splitlines():
                    converted = convert_label_line(line, mapping, source_label)
                    if converted:
                        converted_lines.append(converted)

            destination_label.write_text("\n".join(converted_lines), encoding="utf-8")
            image_count[out_split] += 1
            label_count += len(converted_lines)

class_names = [""] * len(class_to_id)
for canonical_name, class_id in class_to_id.items():
    class_names[class_id] = class_display_names[canonical_name]

if image_count["train"] == 0:
    raise RuntimeError("No training images were found in either dataset.")
if image_count["val"] == 0:
    raise RuntimeError("No validation images were found. Check the downloaded dataset splits.")

data_yaml = {
    "path": str(output),
    "train": "train/images",
    "val": "val/images",
    "names": class_names,
}
if image_count["test"]:
    data_yaml["test"] = "test/images"

(output / "data.yaml").write_text(
    yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print("Combined data.yaml:", output / "data.yaml")
print("Images by split:", image_count)
print("Converted annotations:", label_count)
print("Number of unified classes:", len(class_names))
print("Classes:", class_names)


## 3. Train the quick YOLO11n baseline

This preset is intentionally small for a quick test. For a stronger model, increase the number of epochs and image size, and evaluate on real photos that were not used for training.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. In Colab, select Runtime > Change runtime type > GPU.")

training_dir = Path("/content/visionlab_training/electronics_iot")
if training_dir.exists():
    shutil.rmtree(training_dir)

model = YOLO("yolo11n.pt")
model.train(
    data="/content/combined_components/data.yaml",
    epochs=3,
    imgsz=320,
    batch=16,
    fraction=0.25,
    workers=2,
    device=0,
    patience=2,
    cache=False,
    plots=False,
    project="/content/visionlab_training",
    name="electronics_iot",
    exist_ok=True,
    verbose=True,
)

best_model = training_dir / "weights" / "best.pt"
if not best_model.is_file():
    raise FileNotFoundError(f"Training completed without the expected checkpoint: {best_model}")

export_model = Path("/content/best.pt")
shutil.copy2(best_model, export_model)

trained_model = YOLO(str(export_model))
print("Training complete.")
print("Exported checkpoint:", export_model)
print("Model classes:", trained_model.names)
print("Checkpoint size (MB):", round(export_model.stat().st_size / (1024 * 1024), 2))


## 4. Download only the model checkpoint

After downloading, upload the file into the repository at `models/best.pt`. The live webcam loader will prefer this checkpoint automatically; if it is absent, the current `models/components.pt` remains the fallback.


In [ ]:
from google.colab import files

files.download("/content/best.pt")
